# YouTube Lecture Synthesizer - Experimentation Notebook

Use this notebook to test and experiment with different components of the RAG pipeline.

In [ ]:
import sys
sys.path.append('../src')

from transcript_extractor import TranscriptExtractor
from chunker import SmartChunker
from vector_store import VectorStore
from rag_engine import RAGEngine
from utils import PlaylistProcessor, TranscriptExporter

## 1. Extract Transcript

In [ ]:
extractor = TranscriptExtractor()

# Try your own video URL
video_url = "https://www.youtube.com/watch?v=aircAruvnKk"

transcript_data = extractor.get_transcript(video_url)

print(f"Video ID: {transcript_data['video_id']}")
print(f"Total segments: {transcript_data['total_segments']}")
print(f"\nFirst 3 segments:")
for seg in transcript_data['segments'][:3]:
    print(f"[{seg['timestamp']}] {seg['text']}")

## 2. Experiment with Chunking

In [ ]:
# Try different chunk sizes
chunker = SmartChunker(chunk_size=500, chunk_overlap=100)

chunks = chunker.chunk_transcript(transcript_data['segments'])

print(f"Created {len(chunks)} chunks")
print(f"\nFirst chunk:")
print(f"Timestamp: {chunks[0]['timestamp_range']}")
print(f"Text: {chunks[0]['text'][:200]}...")

# Get statistics
stats = chunker.get_chunk_stats(chunks)
print(f"\nChunk Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 3. Store and Search

In [ ]:
vector_store = VectorStore()

# Add documents
vector_store.add_documents(
    chunks=chunks,
    video_id=transcript_data['video_id'],
    video_url=transcript_data['video_url']
)

# Test search
query = "What is a neural network?"
results = vector_store.search(query, top_k=3)

print(f"Search results for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"  Timestamp: {result['metadata']['timestamp_range']}")
    print(f"  Text: {result['text'][:150]}...\n")

## 4. RAG Query

In [ ]:
rag_engine = RAGEngine(vector_store)

# Ask questions
questions = [
    "What is a neural network?",
    "How does backpropagation work?",
    "What are activation functions?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    
    result = rag_engine.query(question, top_k=3)
    
    print(f"\nA: {result['answer']}")
    print(f"\nSources:")
    for i, source in enumerate(result['sources'], 1):
        print(f"  {i}. [{source['timestamp_range']}]")

## 5. Export Transcript

In [ ]:
exporter = TranscriptExporter()

# Export as text
text_export = exporter.to_text(transcript_data['segments'][:10], include_timestamps=True)
print("Text Export (first 10 segments):")
print(text_export)

# Save to file
with open('../data/transcript_export.txt', 'w', encoding='utf-8') as f:
    f.write(exporter.to_text(transcript_data['segments']))
print("\n✅ Full transcript saved to data/transcript_export.txt")

## 6. Video Summary

In [ ]:
summary = rag_engine.summarize_video(transcript_data['video_id'], max_chunks=10)
print("Video Summary:")
print(summary)